# Exercises: Stacks and Queues

We had the lecture on stacks and queues but no exercises to go with it - these fill that gap.

Quick recap before diving in:
- A **stack** is LIFO (Last In, First Out) - think a stack of plates, you take from the top.
- A **queue** is FIFO (First In, First Out) - think a line at a store, first person in line gets served first.

## Exercise 1: Implement a Stack

Build a stack backed by a plain Python list. `push` adds to the top, `pop` removes and returns the top, `peek` looks at the top without removing it.

In [1]:
class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)          # add to the END of the list = the "top" of the stack

    def pop(self):
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self.items.pop()          # removes and returns the LAST item - O(1), no shifting needed

    def peek(self):
        if self.is_empty():
            raise IndexError("peek from empty stack")
        return self.items[-1]

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

In [2]:
s = Stack()
s.push(1)
s.push(2)
s.push(3)
print("stack size:", s.size())
print("peek:", s.peek())      # should be 3 - the last thing pushed
print("pop:", s.pop())        # 3 comes back out first (LIFO)
print("pop:", s.pop())        # then 2
print("is_empty:", s.is_empty())

stack size: 3
peek: 3
pop: 3
pop: 2
is_empty: False


Note why the list's END is used as the "top" rather than the front: `.append()` and `.pop()` (no argument) both operate on the last element and are O(1). If we used the front instead, every push/pop would mean shifting every other element - the same shifting cost we ran into with arrays back when comparing them to linked lists.

## Exercise 2: Implement a Queue

Same idea, opposite discipline: `enqueue` adds to the back, `dequeue` removes from the front.

In [3]:
class Queue:
    def __init__(self):
        self.items = []

    def enqueue(self, item):
        self.items.append(item)       # add to the back

    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        return self.items.pop(0)      # remove from the FRONT - but this is O(n)!

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

In [4]:
q = Queue()
q.enqueue("a")
q.enqueue("b")
q.enqueue("c")
print("dequeue:", q.dequeue())   # "a" comes out first (FIFO)
print("dequeue:", q.dequeue())   # then "b"
print("size:", q.size())

dequeue: a
dequeue: b
size: 1


**Watch out**: `self.items.pop(0)` removing the front of a list means every remaining element has to shift down one slot to fill the gap - that's the exact same array-shifting cost we talked about with the "insert into 4th place" race example, just happening on every single `dequeue()` call instead of an occasional insert.

The fix: use `collections.deque` instead of a plain list. A deque is built (internally, something like a doubly linked list of blocks) so that adding/removing from *either* end is O(1) - no shifting.

In [5]:
from collections import deque

class FastQueue:
    def __init__(self):
        self.items = deque()

    def enqueue(self, item):
        self.items.append(item)        # O(1)

    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        return self.items.popleft()    # O(1) - no shifting, unlike list.pop(0)

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

In [6]:
fq = FastQueue()
fq.enqueue(1)
fq.enqueue(2)
fq.enqueue(3)
print("fast dequeue:", fq.dequeue())

fast dequeue: 1


## Exercise 3: Balanced parentheses (classic stack application)

Write a function that checks whether every opening bracket in a string has a matching, correctly-ordered closing bracket. This is one of the most common real uses of a stack.

The idea: walk through the string. Every time you see an opening bracket, push it. Every time you see a closing bracket, pop the stack and check that it matches. If the stack ever comes up empty when you expect something to pop, or the popped bracket doesn't match, the string isn't balanced. At the very end, the stack should be empty too - otherwise something was opened and never closed.

In [7]:
def is_balanced(expression):
    stack = Stack()
    pairs = {')': '(', ']': '[', '}': '{'}

    for ch in expression:
        if ch in '([{':
            stack.push(ch)
        elif ch in ')]}':
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False

    return stack.is_empty()   # if anything is left un-closed, it's not balanced


tests = ["(a+b)*(c-d)", "([)]", "{[()]}", "(()", "no brackets here"]
for t in tests:
    print(f"{t!r} -> {is_balanced(t)}")

'(a+b)*(c-d)' -> True
'([)]' -> False
'{[()]}' -> True
'(()' -> False
'no brackets here' -> True


`"([)]"` is a good one to trace by hand: the brackets ARE all present and even paired up in count, but `]` shows up while `(` is still open on the stack (not `[`) - order matters, not just matching counts. That's exactly the kind of case this function catches that a naive "count the brackets" approach would miss.

## Exercise 4: Reverse a queue using a stack

A nice exercise that combines both structures: given a queue, reverse the order of its elements, using only a stack as a helper (no peeking at the list underneath).

The trick: dequeue everything into a stack (this alone reverses the order, since a stack is LIFO), then pop everything from the stack back into the queue.

In [8]:
def reverse_queue(q):
    stack = Stack()
    while not q.is_empty():
        stack.push(q.dequeue())     # draining the queue into the stack reverses the order
    while not stack.is_empty():
        q.enqueue(stack.pop())      # draining the stack back into the queue keeps that reversed order
    return q


q2 = Queue()
for x in [1, 2, 3, 4, 5]:
    q2.enqueue(x)

reverse_queue(q2)

result = []
while not q2.is_empty():
    result.append(q2.dequeue())
print("reversed queue:", result)

reversed queue: [5, 4, 3, 2, 1]


Worth noticing: this only works because a stack flips order (LIFO) while a queue preserves it (FIFO) - pushing `1,2,3,4,5` onto a stack and popping it back off gives you `5,4,3,2,1` for free. That's the whole trick, and it's a good example of picking a data structure specifically *because* of its ordering behavior, not just for storage.

## Exercise 5: Palindrome Checking (stack application)

In [9]:
def is_palindrome(text):
    """Use a stack to reverse the text, then just compare - the classic stack trick."""
    cleaned = text.lower().replace(" ", "")

    stack = Stack()
    for ch in cleaned:
        stack.push(ch)

    reversed_text = ""
    while not stack.is_empty():
        reversed_text += stack.pop()

    return cleaned == reversed_text


# Testing checklist straight from the lecture slide: single char, a word, multiple words,
# mixed case, even length, odd length, and the empty string.
tests = [
    "a",
    "racecar",
    "hello",
    "A man a plan a canal Panama",
    "Was it a car or a cat I saw",
    "abba",
    "abcba",
    "",
]
for t in tests:
    print(f"{t!r:35} -> {is_palindrome(t)}")

'a'                                 -> True
'racecar'                           -> True
'hello'                             -> False
'A man a plan a canal Panama'       -> True
'Was it a car or a cat I saw'       -> True
'abba'                              -> True
'abcba'                             -> True
''                                  -> True


Pushing every character onto a stack and then popping them all back off hands you the
string **backwards** for free - that's the whole trick, `Stack.pop()` is a LIFO, so the last
character pushed (the end of the string) is the first one out. Comparing that reversed version
against the (lowercased, space-stripped) original is then a one-line check.

Running through the lecture's testing checklist: single characters and the empty string are
trivially palindromes (nothing to contradict), `"hello"` correctly fails, and the phrase examples
show why `.lower()` and stripping spaces both matter - without them, `"Was it a car or a cat I saw"`
would fail on the capital `W` alone, even though the *letters* read the same forwards and
backwards.

## Exercise 6: Evaluating Postfix Expressions (stack application)

In [10]:
def evaluate_postfix(expression):
    """Follow the lecture's algorithm exactly: numbers get pushed, operators pop two and push
    the result back."""
    stack = Stack()

    for token in expression.split():
        if token.lstrip("-").isdigit():
            stack.push(int(token))
        else:
            right = stack.pop()   # NOTE: right operand was pushed LAST, so it comes off FIRST
            left = stack.pop()
            if token == "+":
                result = left + right
            elif token == "-":
                result = left - right
            elif token == "*":
                result = left * right
            elif token == "/":
                result = left / right
            else:
                raise ValueError(f"unknown operator: {token}")
            stack.push(result)

    return stack.pop()


# The exact worked examples from the "Additional Stack Applications" slide
examples = {
    "4 7 *": 28,
    "4 7 2 + *": 36,
    "4 7 * 20 -": 8,
    "3 4 7 * 2 / +": 17,
}
for postfix, expected in examples.items():
    actual = evaluate_postfix(postfix)
    print(f"{postfix:16} -> {actual:>5}   (expected {expected}, matches={actual == expected})")

4 7 *            ->    28   (expected 28, matches=True)
4 7 2 + *        ->    36   (expected 36, matches=True)
4 7 * 20 -       ->     8   (expected 8, matches=True)
3 4 7 * 2 / +    ->  17.0   (expected 17, matches=True)


All four match the lecture's worked trace exactly - `"4 7 * 20 -"` is the one the slides
walk through step by step (push 4, push 7, `*` pops 7 then 4 and pushes 28, push 20, `-` pops 20 then
28 and pushes 8).

The comment on `right`/`left` matters more than it looks: for `+` and `*` the order doesn't change
the answer, but for `-` and `/` it does. Since a stack is LIFO, whichever operand got pushed *second*
(the right-hand one in the original expression) is always the one that pops off *first* - get that
backwards and `"4 7 * 20 -"` would compute `20 - 28 = -8` instead of the correct `28 - 20 = 8`.

No need for parentheses or operator-precedence rules here either, which is exactly the appeal of
postfix notation the slides mention: by the time you see an operator, both of its operands are
already sitting right there on top of the stack.